# Finetuning a Convolutional Neural Network

In this exercise, you will have to finetune a pretrained CNN model on the CIFAR10 dataset. The data loading and model testing logic are already included in your code. You will have to create the model and the training loop.

**In this workspace you have GPU to help train the model but it is best practice to DISABLE it while writing code and only ENABLE it when you are training.** 

Here are the steps you need to do to complete this exercise:

1. Finish the `create_model()` function. You should use a pretrained model. You are free to choose any pre-trained model that you want to use. 
2. Finish the `train()` function. This function should validate the accuracy of the model during the training stage. You should stop the training when this validation accuracy stops increasing.
3. Save all your work and then **ENABLE** the GPU
4. Run the file to make sure that the model is training properly.
5. If it works, remember to **DISABLE** the GPU before moving to the next page. 

In case you get stuck, you can look at the solution by clicking the jupyter symbol at the top left and navigating to `finetune_a_cnn_solution.py`.

## Try It Out!
- See how your accuracy changes when using other pre-trained models.
- Play around with the number of layers and neurons in your model. How does the accuracy change? How long does it take to train the model?
- Can you create the same network in TensorFlow as well?

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
def test(model, test_loader, criterion, device):
    # Evaluation mode: layers like Dropout and BatchNorm behave as in inference
    model.eval()

    running_loss = 0       # total loss over all test images
    running_corrects = 0   # number of correct predictions

    # No gradients needed: we're only measuring, not learning
    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move the batch to the same device as the model
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass: raw scores, shape [batch_size, 10]
            outputs = model(inputs)

            # Average loss per image in this batch
            loss = criterion(outputs, labels)

            # Predicted class = index of the highest score
            preds = outputs.argmax(dim=1)

            # Average batch loss × number of images = total loss for this batch
            running_loss += loss.item() * inputs.size(0)

            # Count the correct predictions in this batch
            running_corrects += (preds == labels).sum().item()

    # Divide by the number of images, not the number of batches
    n = len(test_loader.dataset)
    total_loss = running_loss / n
    total_acc = 100 * running_corrects / n

    print(f"Test: Loss {total_loss:.4f}, Accuracy {total_acc:.2f}%")
    return total_loss, total_acc

In [3]:
def train(model, train_loader, validation_loader, cost, optimizer, device, epochs=50, patience=3):
    # Best validation loss seen so far. Starts at infinity so the first epoch always counts as an improvement
    best_loss = float("inf")
    # Number of epochs in a row where the validation loss did not improve
    loss_counter = 0
    # Dictionary to pick the right DataLoader for each phase
    loaders = {"train": train_loader, "valid": validation_loader}
    
    for epoch in range(epochs):
        # Each epoch has 2 phases: first train, then validate
        for phase in ["train", "valid"]:
            if phase == "train":
                # Training mode: layers like Dropout and BatchNorm behave as in training
                model.train()
            else:
                # Evaluation mode: those layers behave as in inference
                model.eval()

            # Reset the counters at the start of each phase
            running_loss = 0       # total loss over all images in this phase
            running_corrects = 0   # number of correct predictions in this phase

            # Pick train_loader or validation_loader depending on the phase
            current_loader = loaders[phase]

            # Go through the data one batch at a time
            for inputs, labels in current_loader:

                # Move the batch to the same device as the model (CPU or GPU)
                inputs, labels = inputs.to(device), labels.to(device)

                # Track gradients only during training (saves memory and time during validation)
                with torch.set_grad_enabled(phase == "train"):

                    # Forward pass: get the model's raw scores (logits) for each class
                    outputs = model(inputs)

                    # Measure how wrong the predictions are (average loss per image in the batch)
                    loss = cost(outputs, labels)

                    # Only update the weights during training, never during validation
                    if phase == "train":
                        optimizer.zero_grad()   # clear the gradients from the previous batch
                        loss.backward()         # calculate the gradients (backpropagation)
                        optimizer.step()        # update the weights using those gradients

                # Predicted class = index of the highest score for each image
                preds = outputs.argmax(dim=1)

                # Average batch loss × number of images = total loss for this batch
                running_loss += loss.item() * inputs.size(0)

                # Count how many predictions in this batch are correct
                running_corrects += (preds == labels).sum().item()

            # Total number of images in this phase's dataset
            n = len(current_loader.dataset)

            # Average loss per image for this phase
            epoch_loss = running_loss / n

            # Percentage of correct predictions for this phase
            epoch_acc = 100 * running_corrects / n

            # Show the results for this phase
            print(f"Epoch {epoch+1}, {phase}: Loss {epoch_loss:.4f}, Accuracy {epoch_acc:.2f}%")

            # Early stopping check, based only on the validation results
            if phase == "valid":
                if epoch_loss < best_loss:
                    # The validation loss improved: save it and reset the counter
                    best_loss = epoch_loss
                    loss_counter = 0
                else:
                    # No improvement: increase the counter
                    loss_counter += 1

        # Stop training if the validation loss hasn't improved for `patience` epochs in a row
        if loss_counter >= patience:
            print(f"Early stopping: validation loss did not improve for {patience} epochs")
            break

    # Return the trained model
    return model

In [4]:
def create_model():
    # TODO: Write code to create the model
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Linear(num_features, 10))
    
    return model

In [5]:
batch_size=32
training_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

testing_transform = transforms.Compose([transforms.ToTensor(),
    transforms.RandomResizedCrop(224),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
        download=True, transform=training_transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
        shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
        download=True, transform=testing_transform)

testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
        shuffle=False)

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Running on Device {device}")

epochs = 2

model=create_model()
model=model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

Running on Device cuda:0


In [7]:
train(model, trainloader, testloader, criterion, optimizer, device, epochs)

Epoch 1, train: Loss 0.7974, Accuracy 73.40%
Epoch 1, valid: Loss 1.5024, Accuracy 49.99%
Epoch 2, train: Loss 0.6416, Accuracy 77.89%
Epoch 2, valid: Loss 1.6137, Accuracy 50.84%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [9]:
test(model, testloader, criterion, device)

Test: Loss 1.5946, Accuracy 51.21%


(1.5945534343719483, 51.21)

## Remember to Disable GPU when you are done training. 